In [20]:
!pip install sklearn-pandas pandas scikit-learn

In [21]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn_pandas import DataFrameMapper

URL_BASE = "https://raw.githubusercontent.com/FranciscoAmbort/grupo13_castellano_ambort_ia_2026/main/entrega_3/data"

url_train = f"{URL_BASE}/solicitudes_train.csv"
url_test = f"{URL_BASE}/solicitudes_test.csv"
url_comp = f"{URL_BASE}/comportamiento_bancario.csv"

# 2. Carga directa optimizada
df_sol_train = pd.read_csv(url_train)
df_sol_test = pd.read_csv(url_test)
df_comportamiento = pd.read_csv(url_comp)

In [22]:
# En este intento realizamos lo siguiente: calcular el retorno esperado por dólar prestado combinando la probabilidad de pago del modelo con la tasa de interés y el costo de default, filtrando únicamente
#las operaciones con rentabilidad esperada positiva. Luego, ordena a los solicitantes de mayor a menor retorno y aprueba los créditos hasta agotar el presupuesto límite.

In [23]:
df_banco_sin_mes = df_comportamiento.drop(columns=['Mes_Referencia'])
comportamiento_resumen = df_banco_sin_mes.groupby('ID_Usuario').mean().reset_index()

#Quitamos la fecha de solicitud
df_sol_train_limpio = df_sol_train.drop(columns=['Fecha_Solicitud'])
df_sol_comp = df_sol_train_limpio.merge(comportamiento_resumen, on= 'ID_Usuario', how= 'left')
df_sol_comp.head()


df_sol_comp = df_sol_comp.drop(columns=['ID_Usuario'])


# Intento retorno esperado por dolar


In [24]:
mapper_sol_comp = DataFrameMapper([
   (['Monto_Solicitado'], [StandardScaler()]),
   (['Tasa_Interes'], [StandardScaler()]),
   (['Ingreso_Mensual'], [StandardScaler()]),
   (['Motivo_Prestamo'], [OneHotEncoder()]),
   (['Dias_Atraso_Otras_Deudas'], [StandardScaler()])
])


lr_model_full = Pipeline([
    ('mapper', mapper_sol_comp),
    ('classifier', LogisticRegression(random_state=42)),
])

lr_model_full.fit(df_sol_comp, df_sol_comp.Estado_Final)

test_eval_roi = df_sol_test.merge(comportamiento_resumen, on='ID_Usuario', how='left').copy()
test_eval_roi['prob_paga'] = lr_model_full.predict_proba(test_eval_roi)[:, 1]

#Calculamos el Retorno Esperado por dólar prestado
test_eval_roi['retorno_esperado'] = (
    test_eval_roi['prob_paga'] * test_eval_roi['Tasa_Interes']
    - (1 - test_eval_roi['prob_paga']) * 1.0
)

# Filtramos solo los que tienen expectativa positiva de ganancia
#    y ordenamos de + a - retorno esperado
candidatos_roi = test_eval_roi[test_eval_roi['retorno_esperado'] > 0].sort_values(
    by='retorno_esperado',
    ascending=False
)

PRESUPUESTO_MAX = 2_000_000
monto_acumulado = 0
ids_aprobados_roi = []

for idx, fila in candidatos_roi.iterrows():
    monto = fila['Monto_Solicitado']
    if monto_acumulado + monto <= PRESUPUESTO_MAX:
        monto_acumulado += monto
        ids_aprobados_roi.append(fila['ID_Solicitud'])
    else:
        continue

test_eval_roi['Decision_Aprobacion'] = test_eval_roi['ID_Solicitud'].isin(ids_aprobados_roi).astype(int)

submission_roi = test_eval_roi[['ID_Solicitud', 'Decision_Aprobacion']]
submission_roi.to_csv('submission_retorno_esperado.csv', index=False)

print(f"Total registros: {len(submission_roi)}")
print(f"Créditos aprobados: {submission_roi['Decision_Aprobacion'].sum()}")
print(f"Monto total colocado: ${monto_acumulado:,.2f} USD")

/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function tosequence is deprecated; tosequence was deprecated in 1.5 and will be removed in 1.7
  warnings.warn(msg, category=FutureWarning)


Total registros: 3061
Créditos aprobados: 552
Monto total colocado: $1,999,600.00 USD


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>